# 99 — Energy BVAR result inspector

Permanent **read-only** audit notebook for the whole `bvar-energy/results` tree.

It can inspect:

- any component BVAR run;
- any forecast contract (`unconditional`, `conditional`, or future names);
- component HICP forecast products;
- the full posterior `draws.npz`;
- the fitted-path cache `fit_draws.npz`;
- Parquet/CSV summaries and diagnostics;
- HICP Energy aggregate runs.

The notebook never modifies a run, never promotes a run, and never selects by filesystem modification time.

## 1. Selection

Two modes are supported.

**Auto mode:** select a model and optionally a vintage/run. The resolver prefers the **promoted SQLite run**. If there is no promoted registry row, it uses `created_at` stored in metadata.

**Path mode:** paste a path to a run directory, a forecast directory, or even a file inside one of those directories.

Examples:

```python
MODE = "auto"
MODEL = "gas"
VINTAGE = None
RUN_ID = None
FORECAST = None
```

or

```python
MODE = "path"
PATH = r"C:\...\results\gas\20260809\<run_id>"
```

or directly a forecast:

```python
PATH = r"C:\...\results\gas\20260809\<run_id>\forecasts\conditional"
```

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------------
# USER CONTROLS — normally this is the only cell you edit.
# ------------------------------------------------------------------
MODE = "auto"          # "auto" or "path"

# Auto mode
MODEL = "gas"          # gas, electricity, heat_energy, solid_fuels,
                       # petrol, diesel, liquid_fuels, aggregate
VINTAGE = None         # e.g. "20260809"; None = latest usable vintage
RUN_ID = None          # exact run id, or None
FORECAST = None        # e.g. "unconditional", "conditional"; None = run-level view
PREFER_PROMOTED = True

# Path mode
PATH = None            # paste any path inside a result store

# Optional comparison settings
COMPARE_FORECASTS = None   # e.g. ["unconditional", "conditional"]; None = all valid
COMPARE_PRODUCT = "hicp_yoy"  # "hicp_yoy", "hicp_level", or "level:<variable>"
COMPARE_INTERVAL = 0.68


In [ ]:
def _find_project_root(start=None):
    base = Path.cwd() if start is None else Path(start)
    base = base.expanduser().resolve()
    for candidate in [base, *base.parents]:
        if (candidate / "src").exists() and (candidate / "results").exists():
            return candidate
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from inside "
        "bvar-energy or set PROJECT_ROOT manually in this cell."
    )

PROJECT_ROOT = _find_project_root()
ANALYSIS_DIR = PROJECT_ROOT / "src" / "analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from energy_bvar_inspection import (
    AGGREGATE_MODEL_ID,
    InspectionError,
    compare_forecasts,
    discover_forecasts,
    file_inventory,
    inspect_json,
    load_table,
    npz_manifest,
    plot_forecast_comparison,
    resolve_result,
    result_health,
    run_overview,
    summarize_npz_array,
    table_manifest,
)

print("Project root:", PROJECT_ROOT)


In [ ]:
if MODE == "path":
    if not PATH:
        raise ValueError("MODE='path' requires PATH.")
    ctx = resolve_result(project_root=PROJECT_ROOT, path=PATH, forecast_name=FORECAST)
elif MODE == "auto":
    ctx = resolve_result(
        project_root=PROJECT_ROOT,
        model=MODEL,
        vintage=VINTAGE,
        run_id=RUN_ID,
        forecast_name=FORECAST,
        prefer_promoted=PREFER_PROMOTED,
    )
else:
    raise ValueError("MODE must be 'auto' or 'path'.")

display(ctx.as_frame())


## 2. Run overview and health checks

In [ ]:
display(run_overview(ctx))
display(result_health(ctx))


## 3. File inventory

This is the fastest way to understand what was actually persisted.  
`draws.npz` is the full posterior used by Structural analysis; `fit_draws.npz` is the compact fitted-path cache.

In [ ]:
inventory = file_inventory(ctx)
display(inventory)


## 4. Metadata / configuration

In [ ]:
json_files = [
    ctx.run_directory / "metadata.json",
    ctx.run_directory / "aggregate_config.json",
]
json_files = [p for p in json_files if p.is_file()]

if not json_files:
    print("No run-level JSON metadata/config file found.")
else:
    for path in json_files:
        print(f"\n### {path.name}")
        display(inspect_json(path))


## 5. Parquet and CSV tables

Tables stay in their efficient on-disk format; this section renders them as ordinary pandas DataFrames.

In [ ]:
table_files = []
for rel in inventory.loc[inventory["type"].isin(["parquet", "csv"]), "relative_path"]:
    path = ctx.run_directory / rel
    # Keep the default notebook compact: display the main audit tables automatically.
    if path.name in {
        "summary.parquet", "summary.csv",
        "diagnostics.parquet", "diagnostics.csv",
        "history.parquet", "history.csv",
    }:
        table_files.append(path)

for path in table_files:
    print(f"\n### {path.relative_to(ctx.run_directory)}")
    try:
        frame = load_table(path)
        print("shape:", frame.shape)
        display(frame)
    except Exception as exc:
        print("Could not load:", exc)


### Inspect any other table manually

Change `TABLE_PATH` to any `.parquet` or `.csv` listed in the inventory, including a `display_v1.parquet` inside a forecast store.

In [ ]:
TABLE_PATH = None  # e.g. ctx.run_directory / "forecasts/unconditional/display_v1.parquet"

if TABLE_PATH is not None:
    TABLE_PATH = Path(TABLE_PATH)
    display(table_manifest(TABLE_PATH))
    display(load_table(TABLE_PATH))
else:
    print("Set TABLE_PATH when you want to inspect a specific table.")


## 6. NPZ manifests

The manifest reads **array names, shapes and dtypes without dumping the arrays**.  
This is the safe first step for `draws.npz`, `forecast_draws.npz`, `hicp_draws.npz`, and `aggregate_draws.npz`.

In [ ]:
npz_files = [
    ctx.run_directory / rel
    for rel in inventory.loc[inventory["type"].eq("npz"), "relative_path"]
]

for path in npz_files:
    print(f"\n### {path.relative_to(ctx.run_directory)}")
    display(npz_manifest(path))


### Numerically inspect one NPZ array

Example:

```python
NPZ_PATH = ctx.run_directory / "draws.npz"
ARRAY = "B"
```

For very large arrays the helper samples at most two million values for descriptive statistics; the original shape is always reported.

In [ ]:
NPZ_PATH = None
ARRAY = None

if NPZ_PATH is not None and ARRAY is not None:
    display(summarize_npz_array(NPZ_PATH, ARRAY).to_frame("value"))
else:
    print("Set NPZ_PATH and ARRAY to inspect one array numerically.")


## 7. Forecast contracts

Every directory under `forecasts/` is discovered dynamically. Nothing is hard-coded to `unconditional`.

In [ ]:
if ctx.model_id == AGGREGATE_MODEL_ID:
    print("This is an aggregate run; component forecast contracts live in the component runs.")
    forecasts = pd.DataFrame()
else:
    forecasts = discover_forecasts(ctx.run_directory)
    display(forecasts)


### Inspect each forecast store

For every discovered forecast, show its forecast metadata, HICP metadata when present, and the NPZ schemas.

In [ ]:
if ctx.model_id != AGGREGATE_MODEL_ID and len(forecasts):
    for _, row in forecasts.iterrows():
        fdir = Path(row["directory"])
        print("\n" + "=" * 90)
        print("FORECAST:", row["forecast_name"])
        print("=" * 90)

        for name in ("forecast_metadata.json", "hicp_metadata.json"):
            path = fdir / name
            if path.is_file():
                print(f"\n### {name}")
                display(inspect_json(path))

        for name in ("forecast_draws.npz", "hicp_draws.npz"):
            path = fdir / name
            if path.is_file():
                print(f"\n### {name}")
                display(npz_manifest(path))
else:
    print("No component forecast stores to inspect.")


## 8. Compare conditional / unconditional / other forecasts

If at least two compatible forecast stores exist, the notebook can compare them without knowing their names in advance.

`COMPARE_FORECASTS = None` means **all valid stores**.

Supported products:

- `hicp_yoy`
- `hicp_level`
- `level:<variable>`, for example `level:gas_pre_tax`


In [ ]:
comparison = None

if ctx.model_id == AGGREGATE_MODEL_ID:
    print("Forecast comparison here is for component runs. Aggregate scenarios are inspected from aggregate_draws.npz.")
elif len(forecasts) < 2 and COMPARE_FORECASTS is None:
    print("Only one forecast contract is available; nothing to compare yet.")
else:
    try:
        comparison = compare_forecasts(
            ctx.run_directory,
            forecast_names=COMPARE_FORECASTS,
            product=COMPARE_PRODUCT,
            interval=COMPARE_INTERVAL,
        )
        display(comparison.head(20))
        display(
            comparison.groupby("forecast_name")
            .agg(
                first_date=("date", "min"),
                last_date=("date", "max"),
                n_draws=("n_draws", "max"),
                terminal_median=("median", "last"),
                terminal_lower=("lower", "last"),
                terminal_upper=("upper", "last"),
            )
        )
        plot_forecast_comparison(
            comparison,
            title=f"{COMPARE_PRODUCT} — forecast comparison",
        )
    except Exception as exc:
        print("Comparison unavailable:", exc)


## 9. Aggregate-run quick inspection

When `MODEL="aggregate"` (or when a path points to an aggregate run), this section gives a compact numerical summary of the saved aggregate arrays.

In [ ]:
if ctx.model_id == AGGREGATE_MODEL_ID:
    aggregate_npz = ctx.run_directory / "aggregate_draws.npz"
    if aggregate_npz.is_file():
        manifest = npz_manifest(aggregate_npz)
        display(manifest)
        for key in (
            "baseline_level_paths",
            "baseline_yoy_paths",
            "scenario_level_paths",
            "scenario_yoy_paths",
            "baseline_contribution_paths",
        ):
            if key in manifest.index:
                print(f"\n### {key}")
                display(summarize_npz_array(aggregate_npz, key).to_frame("value"))
    else:
        print("aggregate_draws.npz not found.")
else:
    print("Current selection is a component run.")


## 10. Raw access when needed

Everything above is read-only convenience. You can always load the underlying formats directly:

```python
# JSON
import json
metadata = json.loads((ctx.run_directory / "metadata.json").read_text())

# Parquet
summary = pd.read_parquet(ctx.run_directory / "summary.parquet")

# NPZ
with np.load(ctx.run_directory / "draws.npz", allow_pickle=False) as z:
    B = z["B"]
```

Use the manifest first so you know exactly what you are loading.